# r/gradadmissions — September 2024 Topic + Sentiment Analysis

This notebook runs **Topic Modeling** and **Sentiment Analysis** separately for the **September 2024** dataset from **r/gradadmissions**.

## What you'll get
### Topic modeling (LDA)
- Preprocess text (title + body)
- Train LDA (e.g., 5–6 topics)
- Visualize topic prevalence
- Inspect top words per topic (so you can label topics)

### Sentiment analysis (VADER)
- Compute sentiment scores (compound ∈ [-1, 1])
- Plot sentiment distribution and positive/neutral/negative share
- Identify **which topic is most negative** (avg sentiment + % negative)

---
## Expected input
A CSV (or JSONL) containing Reddit posts with at least:
- `created_utc` (timestamp or datetime)
- `title`
- `selftext` (post body)

If your column names differ, adjust in **Step 1**.


## 0) Install/Import

If you run this in a fresh environment, you may need to install VADER:

```bash
pip install vaderSentiment
```

If you're on Colab:

```bash
!pip install vaderSentiment
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

import re
from datetime import datetime, timezone

pd.set_option('display.max_colwidth', 200)


## 1) Load your dataset

Set the path to your September 2024 dataset file.

- If your file contains **multiple months**, this notebook will filter for **September 2024**.
- If your file is already only September 2024, filtering will keep everything.


In [ ]:
# TODO: Update this path
DATA_PATH = 'r_gradadmissions_posts.jsonl'   # <-- change filename

# If JSONL
df = pd.read_json(DATA_PATH, lines=True)

# ---- Column mapping (adjust if needed) ----
# Expected columns: created_utc (timestamp), title, selftext
CREATED_COL = 'created_utc'
TITLE_COL = 'title'
BODY_COL = 'selftext'

print('Rows:', len(df))
print('Columns:', df.columns.tolist())
df.head(3)

## 2) Filter to September 2024

This handles:
- Unix timestamps (seconds)
- ISO datetime strings


In [ ]:
def to_datetime_utc(x):
    """Convert created_utc to timezone-aware UTC datetime."""
    if pd.isna(x):
        return pd.NaT
    # If numeric, assume unix seconds
    if isinstance(x, (int, float, np.integer, np.floating)):
        try:
            return datetime.fromtimestamp(float(x), tz=timezone.utc)
        except Exception:
            return pd.NaT
    # If string, try parse
    try:
        dt = pd.to_datetime(x, utc=True, errors='coerce')
        if pd.isna(dt):
            return pd.NaT
        # Ensure tz-aware
        if dt.tzinfo is None:
            dt = dt.tz_localize('UTC')
        return dt.to_pydatetime()
    except Exception:
        return pd.NaT

df['created_dt'] = df[CREATED_COL].apply(to_datetime_utc)
df = df.dropna(subset=['created_dt'])

# Filter Sept 2024 inclusive
start = datetime(2024, 9, 1, tzinfo=timezone.utc)
end = datetime(2024, 10, 1, tzinfo=timezone.utc)
sept = df[(df['created_dt'] >= start) & (df['created_dt'] < end)].copy()

print('September 2024 rows:', len(sept))
sept[['created_dt', TITLE_COL, BODY_COL]].head(3)


## 3) Prepare text field (title + body)

We create one text column used for both topic modeling and sentiment.


In [ ]:
sept[TITLE_COL] = sept[TITLE_COL].fillna('')
sept[BODY_COL] = sept[BODY_COL].fillna('')

sept['text_raw'] = (sept[TITLE_COL].astype(str) + ' ' + sept[BODY_COL].astype(str)).str.strip()
sept = sept[sept['text_raw'].str.len() > 0].copy()
# Remove deleted/removed posts
sept = sept[~sept[BODY_COL].isin(['[deleted]', '[removed]'])]

# Remove very short posts (helps LDA)
sept = sept[sept['text_raw'].str.len() > 20]

print('Non-empty text rows:', len(sept))
sept[['text_raw']].head(3)


# PART A — Topic Modeling (LDA)

## A1) Text cleaning (lightweight)

We remove URLs and non-letters. Keep it simple for a first pass.


In [ ]:
url_re = re.compile(r'https?://\S+|www\.\S+')
non_letter_re = re.compile(r'[^a-zA-Z\s]')

def clean_for_topics(s: str) -> str:
    s = s.lower()
    s = url_re.sub(' ', s)
    s = non_letter_re.sub(' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

sept['text_topics'] = sept['text_raw'].astype(str).apply(clean_for_topics)
sept[['text_topics']].head(3)


## A2) Vectorize + Train LDA

- `n_topics`: start with 5 or 6
- `min_df` / `max_df`: helps remove extremely rare/common terms


In [ ]:
n_topics = 5  # TODO: try 5 or 6

vectorizer = CountVectorizer(
    stop_words='english',
    min_df=3,          # was 5
    max_df=0.90,       # was 0.95
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(sept['text_topics'])
lda = LatentDirichletAllocation(
    n_components=n_topics,
    random_state=42,
    learning_method='batch'
)
topic_dist = lda.fit_transform(X)  # rows: docs, cols: topics

sept['topic_id'] = topic_dist.argmax(axis=1)
sept['topic_conf'] = topic_dist.max(axis=1)

sept[['topic_id', 'topic_conf']].head(5)


## A3) Inspect top words per topic

Use this table to manually label topics (e.g., "Funding", "SOP", "GRE", "GPA", "International", "Interviews").


In [ ]:
def top_words_per_topic(model, feature_names, n_top_words=12):
    rows = []
    for topic_idx, topic in enumerate(model.components_):
        top_idx = topic.argsort()[:-n_top_words-1:-1]
        words = [feature_names[i] for i in top_idx]
        rows.append({'topic_id': topic_idx, 'top_terms': ', '.join(words)})
    return pd.DataFrame(rows)

topic_terms = top_words_per_topic(lda, vectorizer.get_feature_names_out(), n_top_words=12)
topic_terms


## A4) Visualize topic prevalence

Bar chart: which topics are most common in September 2024?


In [ ]:
topic_counts = sept['topic_id'].value_counts().sort_index()

plt.figure(figsize=(10, 5))
plt.bar(topic_counts.index.astype(str), topic_counts.values)
plt.title('Topic distribution — r/gradadmissions (Sept 2024)')
plt.xlabel('Topic ID')
plt.ylabel('# posts')
plt.show()

topic_counts.to_frame('count')


# PART B — Sentiment Analysis (VADER)

## B1) Compute sentiment (compound ∈ [-1, 1])

- compound > 0.05 → positive
- compound < -0.05 → negative
- otherwise → neutral


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def vader_compound(s: str) -> float:
    if not isinstance(s, str) or len(s.strip()) == 0:
        return np.nan
    return analyzer.polarity_scores(s)['compound']

sept['sent_compound'] = sept['text_raw'].astype(str).apply(vader_compound)
sept = sept.dropna(subset=['sent_compound']).copy()

def sentiment_label(c: float) -> str:
    if c > 0.05:
        return 'positive'
    if c < -0.05:
        return 'negative'
    return 'neutral'

sept['sent_label'] = sept['sent_compound'].apply(sentiment_label)
sept[['sent_compound', 'sent_label']].head(5)


## B2) Sentiment range + distribution


In [ ]:
sent_min = sept['sent_compound'].min()
sent_max = sept['sent_compound'].max()
sent_mean = sept['sent_compound'].mean()

print('Sentiment (compound) range:', sent_min, 'to', sent_max)
print('Mean compound:', sent_mean)

plt.figure(figsize=(10, 5))
plt.hist(sept['sent_compound'], bins=30)
plt.title('VADER compound sentiment distribution — Sept 2024')
plt.xlabel('compound score (-1 to 1)')
plt.ylabel('# posts')
plt.show()


## B3) Positive / Neutral / Negative share


In [ ]:
sent_counts = sept['sent_label'].value_counts().reindex(['negative', 'neutral', 'positive']).fillna(0).astype(int)
sent_props = (sent_counts / sent_counts.sum()).round(3)

plt.figure(figsize=(7, 4))
plt.bar(sent_counts.index, sent_counts.values)
plt.title('Sentiment label counts — Sept 2024')
plt.xlabel('sentiment')
plt.ylabel('# posts')
plt.show()

pd.DataFrame({'count': sent_counts, 'proportion': sent_props})


## B4) Which topic is most negative?

We compute for each topic:
- average compound sentiment
- % of posts labeled negative

Then we identify the "most negative" topic.


In [ ]:
topic_sent_summary = sept.groupby('topic_id').agg(
    posts=('topic_id', 'size'),
    avg_compound=('sent_compound', 'mean'),
    pct_negative=('sent_label', lambda x: (x == 'negative').mean())
).reset_index()

topic_sent_summary['pct_negative'] = (topic_sent_summary['pct_negative'] * 100).round(1)
topic_sent_summary['avg_compound'] = topic_sent_summary['avg_compound'].round(3)

topic_sent_summary.sort_values(['avg_compound', 'pct_negative']).reset_index(drop=True)


In [ ]:
# Visualization: topic vs average sentiment
tmp = topic_sent_summary.sort_values('avg_compound')

plt.figure(figsize=(10, 5))
plt.bar(tmp['topic_id'].astype(str), tmp['avg_compound'])
plt.title('Average sentiment by topic — Sept 2024')
plt.xlabel('Topic ID')
plt.ylabel('Average compound sentiment')
plt.show()


## B5) Inspect examples from the most negative topic

This helps you write qualitative support (a few example posts) without quoting too much.


In [ ]:
most_negative_topic = topic_sent_summary.sort_values(['avg_compound', 'pct_negative']).iloc[0]['topic_id']
print('Most negative topic_id:', most_negative_topic)

examples = sept[sept['topic_id'] == most_negative_topic].sort_values('sent_compound').head(10)
examples[[ 'created_dt', TITLE_COL, 'sent_compound', 'sent_label', 'topic_conf']]
